# House Prices: Advanced Regression Techniques
**Pipeline**: 前処理 → CV → 7モデル OOF スタッキング → マルチシード提出

- Public Best: **0.11992**
- 7モデル (Ridge, Lasso, ElasticNet, GBR, XGBoost, LightGBM, CatBoost)
- LOO Target Encoding + ドメイン特徴量 + QOL_Score
- マルチシード OOF スタッキング

## Setup (Colab / Local 自動検出)

In [ ]:
import sys

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in __import__('os').environ
print(f"Environment: {'Colab' if IS_COLAB else 'Local'}")

if IS_COLAB:
    !pip install -q lightgbm xgboost catboost

In [ ]:
import os, json, zipfile, requests
from pathlib import Path

# データパス: Local なら house-prices/data/, Colab なら ./data/
if IS_COLAB:
    DATA_DIR = Path("data")
else:
    # notebook のあるディレクトリ基準
    DATA_DIR = Path(__file__).parent / "data" if "__file__" in dir() else Path("data")

DATA_DIR.mkdir(exist_ok=True)

if not (DATA_DIR / "train.csv").exists():
    # Kaggle KGAT トークンを取得 (Colab: Secrets, Local: ~/.kaggle/kaggle.json)
    KAGGLE_TOKEN = None
    if IS_COLAB:
        try:
            from google.colab import userdata
            KAGGLE_TOKEN = userdata.get('KAGGLE_TOKEN')
        except Exception:
            pass

    if KAGGLE_TOKEN is None:
        kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
        if kaggle_json.exists():
            creds = json.loads(kaggle_json.read_text())
            KAGGLE_TOKEN = creds.get("key")

    if KAGGLE_TOKEN is None:
        raise RuntimeError("Kaggle token not found. Set Colab Secret 'KAGGLE_TOKEN' or place ~/.kaggle/kaggle.json")

    url = "https://www.kaggle.com/api/v1/competitions/data/download-all/house-prices-advanced-regression-techniques"
    headers = {"Authorization": f"Bearer {KAGGLE_TOKEN}"}
    print("Downloading competition data...")
    resp = requests.get(url, headers=headers, stream=True)
    print(f"Status: {resp.status_code}")

    if resp.status_code == 200:
        zip_path = DATA_DIR / "house-prices.zip"
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(DATA_DIR)
        zip_path.unlink()
        print("Data ready!")
    else:
        raise RuntimeError(f"Download failed: {resp.status_code} {resp.text[:500]}")
else:
    print(f"Data already exists at {DATA_DIR}")

# 以降のセルで使うパス
TRAIN_CSV = str(DATA_DIR / "train.csv")
TEST_CSV = str(DATA_DIR / "test.csv")
print(f"TRAIN_CSV: {TRAIN_CSV}\nTEST_CSV: {TEST_CSV}")

## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

## 前処理 (Preprocessor)

In [ ]:
# === ABテスト用フラグ ===
# True: 合成特徴量の元カラムを削除 (冗長性排除)
# False: 元カラムも残す (tree系が個別カラムを活用できる)
DROP_COMPONENTS = False

# 合成特徴量の構成要素 (DROP_COMPONENTS=True のとき削除される)
COMPONENT_COLS = [
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',           # → TotalSF
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath',  # → TotalBath
    'YearBuilt', 'YrSold',                             # → HouseAge, IsNew
    'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch',  # → TotalPorchSF
]


class HousePricesPreprocessor(BaseEstimator, TransformerMixin):
    """前処理: カテゴリカルは object のまま保持し、CatBoost の cat_features に渡す構成"""

    NA_MEANS_NONE = [
        'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
        'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
        'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
        'MasVnrType',
    ]

    ORDINAL_MAP = {
        'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
        'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
        'FireplaceQu': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
        'GarageQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
        'PoolQC': {'None': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
        'Fence': {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
        'CentralAir': {'N': 0, 'Y': 1},
        'Street': {'Grvl': 0, 'Pave': 1},
        'LandSlope': {'Sev': 0, 'Mod': 1, 'Gtl': 2},
        'LotShape': {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3},
    }

    DROP_COLS = ['Id', 'Utilities', 'Street']

    SKEW_FEATURES = [
        'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
        'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
        'GrLivArea', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
        'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal',
    ]

    TE_COLS = ['Neighborhood']
    TE_SMOOTH = 10

    def __init__(self, drop_components=False):
        self.drop_components = drop_components
        self.lot_frontage_medians_ = None
        self.lot_frontage_global_ = None
        self.num_medians_ = None
        self.cat_modes_ = None
        self.feature_names_ = None
        self.cat_feature_names_ = None
        self.cat_feature_indices_ = None
        self.te_stats_ = {}
        self.te_global_mean_ = None
        self._train_y_ = None
        self.first_floor_median_ = None

    def fit(self, X, y=None):
        df = X.copy()
        self.lot_frontage_medians_ = df.groupby('Neighborhood')['LotFrontage'].median()
        self.lot_frontage_global_ = df['LotFrontage'].median()
        num_cols = df.select_dtypes(include=[np.number]).columns
        self.num_medians_ = df[num_cols].median()
        cat_cols = df.select_dtypes(include=['object']).columns
        self.cat_modes_ = df[cat_cols].mode().iloc[0]
        if '1stFlrSF' in df.columns:
            self.first_floor_median_ = df['1stFlrSF'].median()
        if y is not None:
            temp = self._fill_missing(df.copy())
            self._fit_te(temp, y)
        return self

    def fit_transform(self, X, y=None, **fit_params):
        self.fit(X, y)
        self._train_y_ = y
        result = self.transform(X)
        self._train_y_ = None
        return result

    def transform(self, X):
        df = X.copy()
        df = self._fill_missing(df)
        df = self._ordinal_encode(df)
        df = self._apply_te(df)
        df = self._feature_engineering(df)
        df = self._qol_features(df)
        df = self._drop_cols(df)
        if self.drop_components:
            df = self._drop_components(df)
        df = self._fix_skewness(df)

        cat_cols = df.select_dtypes(include=['object']).columns.tolist()
        for col in cat_cols:
            df[col] = df[col].fillna('Missing')
        self.cat_feature_names_ = cat_cols
        self.cat_feature_indices_ = [df.columns.get_loc(c) for c in cat_cols]
        self.feature_names_ = df.columns.tolist()
        return df

    # --- 欠損値処理 ---
    def _fill_missing(self, df):
        for col in self.NA_MEANS_NONE:
            if col in df.columns:
                df[col] = df[col].fillna('None')
        for col in ['GarageYrBlt', 'GarageCars', 'GarageArea']:
            if col in df.columns:
                df[col] = df[col].fillna(0)
        for col in ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                     'BsmtFullBath', 'BsmtHalfBath']:
            if col in df.columns:
                df[col] = df[col].fillna(0)
        if 'LotFrontage' in df.columns:
            for idx in df[df['LotFrontage'].isnull()].index:
                nbhd = df.loc[idx, 'Neighborhood']
                med = self.lot_frontage_medians_.get(nbhd, self.lot_frontage_global_)
                df.loc[idx, 'LotFrontage'] = med
        df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
        num_cols = df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if df[col].isnull().any() and col in self.num_medians_:
                df[col] = df[col].fillna(self.num_medians_[col])
        cat_cols = df.select_dtypes(include=['object']).columns
        for col in cat_cols:
            if df[col].isnull().any() and col in self.cat_modes_:
                df[col] = df[col].fillna(self.cat_modes_[col])
        return df

    def _ordinal_encode(self, df):
        for col, mapping in self.ORDINAL_MAP.items():
            if col in df.columns:
                df[col] = df[col].map(mapping).fillna(0).astype(int)
        return df

    # --- Target Encoding ---
    def _fit_te(self, df, y):
        self.te_global_mean_ = float(y.mean())
        self.te_stats_ = {}
        for col in self.TE_COLS:
            if col in df.columns:
                y_arr = y.values if hasattr(y, 'values') else np.array(y)
                tmp = pd.DataFrame({'cat': df[col].values, 'y': y_arr})
                agg = tmp.groupby('cat')['y'].agg(['sum', 'count'])
                self.te_stats_[col] = {
                    idx: {'sum': row['sum'], 'count': row['count']}
                    for idx, row in agg.iterrows()
                }

    def _apply_te(self, df):
        if self.te_global_mean_ is None:
            return df
        if self._train_y_ is not None:
            return self._te_train(df, self._train_y_)
        return self._te_test(df)

    def _te_train(self, df, y):
        smooth = self.TE_SMOOTH
        gm = self.te_global_mean_
        for col in self.TE_COLS:
            if col not in df.columns or col not in self.te_stats_:
                continue
            y_arr = y.values if hasattr(y, 'values') else np.array(y)
            tmp = pd.DataFrame({'cat': df[col].values, 'y': y_arr})
            cat_sum = tmp.groupby('cat')['y'].transform('sum')
            cat_count = tmp.groupby('cat')['y'].transform('count')
            loo_sum = cat_sum - tmp['y']
            loo_count = cat_count - 1
            df[f'{col}_te'] = ((loo_sum + smooth * gm) / (loo_count + smooth)).values
        return df

    def _te_test(self, df):
        smooth = self.TE_SMOOTH
        gm = self.te_global_mean_
        for col in self.TE_COLS:
            if col not in df.columns or col not in self.te_stats_:
                continue
            encoded = np.full(len(df), gm)
            for cat, stats in self.te_stats_[col].items():
                mask = (df[col] == cat)
                encoded[mask] = (stats['sum'] + smooth * gm) / (stats['count'] + smooth)
            df[f'{col}_te'] = encoded
        return df

    # --- 特徴量エンジニアリング ---
    def _feature_engineering(self, df):
        # 主要合成特徴量
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath']
                           + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df['IsNew'] = (df['YearBuilt'] == df['YrSold']).astype(int)

        # FullBath × TotalSF 交互作用 (グレード×面積の相乗効果)
        df['Bath_x_SF'] = df['FullBath'] * df['TotalSF']

        # 補助合成特徴量
        df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch']
                              + df['3SsnPorch'] + df['ScreenPorch'])
        df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
        df['HasRemod'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
        df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
        df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
        df['Has2ndFlr'] = (df['2ndFlrSF'] > 0).astype(int)
        df['HasPool'] = (df['PoolArea'] > 0).astype(int)
        if self.first_floor_median_ is not None:
            df['Is_SFL'] = ((df['2ndFlrSF'] == 0)
                            & (df['1stFlrSF'] >= self.first_floor_median_)).astype(int)
        df['Living_Space_Ratio'] = df['GrLivArea'] / df['LotArea'].clip(lower=1)
        df['Luxury_Space_Index'] = df['TotalSF'] * df['OverallQual']
        return df

    def _qol_features(self, df):
        df['QOL_Score'] = (
            (df['CentralAir'] == 1).astype(int)
            + (df['GarageArea'] > 0).astype(int)
            + (df['TotalBsmtSF'] > 0).astype(int)
            + (df['Fireplaces'] > 0).astype(int)
            + (df['Functional'] == 8).astype(int)
            + (df['KitchenQual'] >= 4).astype(int)
            + (df['ExterQual'] >= 4).astype(int)
            + (df['BsmtQual'] >= 4).astype(int)
            + (df['FullBath'] >= 2).astype(int)
        )
        return df

    def _drop_cols(self, df):
        drop = [c for c in self.DROP_COLS if c in df.columns]
        return df.drop(columns=drop)

    def _drop_components(self, df):
        """ABテスト: 合成特徴量の構成要素を削除"""
        drop = [c for c in COMPONENT_COLS if c in df.columns]
        return df.drop(columns=drop)

    def _fix_skewness(self, df):
        for feat in self.SKEW_FEATURES:
            if feat in df.columns:
                df[feat] = np.log1p(np.maximum(df[feat], 0))
        for feat in ['TotalSF', 'TotalPorchSF', 'Luxury_Space_Index', 'Bath_x_SF']:
            if feat in df.columns:
                df[feat] = np.log1p(np.maximum(df[feat], 0))
        return df

print(f"HousePricesPreprocessor defined. DROP_COMPONENTS={DROP_COMPONENTS}")

## Pipeline ラッパー
- **CatBoostPipeline**: cat_features を自動渡し (カテゴリカル=object のまま)
- **SklearnPipeline**: object 列を LabelEncode して数値化 (Ridge, GBR 等用)

In [ ]:
class CatBoostPipeline:
    """CatBoost用: cat_features を自動的に渡す"""

    def __init__(self, model):
        self.preprocess = HousePricesPreprocessor(drop_components=DROP_COMPONENTS)
        self.model = model

    def fit(self, X, y):
        X_t = self.preprocess.fit_transform(X, y)
        cat_idx = self.preprocess.cat_feature_indices_
        self.model.fit(X_t, y, cat_features=cat_idx)
        return self

    def predict(self, X):
        X_t = self.preprocess.transform(X)
        return self.model.predict(X_t)


class SklearnPipeline:
    """sklearn用: object 列を LabelEncode して数値化"""

    def __init__(self, model):
        self.preprocess = HousePricesPreprocessor(drop_components=DROP_COMPONENTS)
        self.model = model
        self.label_maps_ = {}

    def fit(self, X, y):
        X_t = self.preprocess.fit_transform(X, y)
        X_t = self._label_encode_fit(X_t)
        self.model.fit(X_t.values.astype(np.float64), y)
        return self

    def predict(self, X):
        X_t = self.preprocess.transform(X)
        X_t = self._label_encode_transform(X_t)
        return self.model.predict(X_t.values.astype(np.float64))

    def _label_encode_fit(self, df):
        self.label_maps_ = {}
        for col in df.select_dtypes(include=['object']).columns:
            vals = sorted(df[col].unique())
            self.label_maps_[col] = {v: i for i, v in enumerate(vals)}
            df[col] = df[col].map(self.label_maps_[col]).fillna(-1).astype(int)
        return df

    def _label_encode_transform(self, df):
        for col, mapping in self.label_maps_.items():
            if col in df.columns:
                df[col] = df[col].map(mapping).fillna(-1).astype(int)
        return df


def make_pipeline(model, use_catboost=False):
    if use_catboost:
        return CatBoostPipeline(model)
    return SklearnPipeline(model)

print(f"CatBoostPipeline & SklearnPipeline defined. DROP_COMPONENTS={DROP_COMPONENTS}")

## データ読み込み

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

# 外れ値除去 (GrLivArea > 4000 & 低価格)
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index
train = train.drop(outliers).reset_index(drop=True)
print(f'外れ値除去: {len(outliers)}件 → Train: {len(train)}行')

y = np.log1p(train['SalePrice'])
X = train.drop(columns=['SalePrice'])
X_test = test.copy()
test_ids = test['Id']

print(f'X: {X.shape}, X_test: {X_test.shape}')

## モデル定義 & CV評価

In [ ]:
def get_models(seed=42):
    """全モデル定義"""
    cb_params = dict(
        iterations=4000 if IS_COLAB else 3000,
        learning_rate=0.03, depth=6,
        l2_leaf_reg=10, od_type='Iter', od_wait=100,
        random_seed=seed, verbose=0,
    )
    if IS_COLAB:
        cb_params['task_type'] = 'GPU'
        cb_params['devices'] = '0'

    return {
        'Ridge': Ridge(alpha=5.0),
        'Lasso': Lasso(alpha=0.0003, max_iter=10000),
        'ElasticNet': ElasticNet(alpha=0.0005, l1_ratio=0.7, max_iter=10000),
        'GBR': GradientBoostingRegressor(
            n_estimators=3000, learning_rate=0.05, max_depth=4,
            min_samples_split=10, min_samples_leaf=5, max_features='sqrt',
            loss='huber', subsample=0.8, random_state=seed),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=2000, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
            reg_lambda=1.0, random_state=seed, verbosity=0),
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=2000, learning_rate=0.05, max_depth=5,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, random_state=seed, verbose=-1),
        'CatBoost': CatBoostRegressor(**cb_params),
    }

CATBOOST_MODELS = {'CatBoost'}
print(f"CatBoost: {'GPU (iter=4000)' if IS_COLAB else 'CPU (iter=3000)'}")

In [ ]:
RMSLE_THRESHOLD = 0.108

def evaluate_models(X, y, n_splits=5, seed=42):
    """KFold CVで複数モデルを評価 (RMSLE = RMSE on log1p target)"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed)

    results = {}
    for name, model in models.items():
        is_cb = name in CATBOOST_MODELS
        rmse_list = []
        for train_idx, val_idx in kf.split(X):
            pipe = make_pipeline(model, use_catboost=is_cb)
            pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = pipe.predict(X.iloc[val_idx])
            rmse_list.append(np.sqrt(mean_squared_error(y.iloc[val_idx], pred)))
        rmse_scores = np.array(rmse_list)
        mean_rmse = rmse_scores.mean()
        results[name] = {
            'mean': mean_rmse,
            'std': rmse_scores.std(),
            'scores': rmse_scores,
        }
        status = '✅' if mean_rmse < RMSLE_THRESHOLD else '  '
        print(f'{status} {name:12s}: RMSLE = {mean_rmse:.5f} ± {rmse_scores.std():.5f}')

    # サマリー
    print(f'\n--- Threshold: {RMSLE_THRESHOLD} ---')
    below = [n for n, r in results.items() if r['mean'] < RMSLE_THRESHOLD]
    if below:
        print(f'✅ {len(below)} model(s) below threshold: {", ".join(below)}')
    else:
        best_val = min(r['mean'] for r in results.values())
        print(f'⚠️  No model below threshold (best: {best_val:.5f}, gap: +{best_val - RMSLE_THRESHOLD:.5f})')

    return results

print('=' * 60)
print('モデル比較 (5-fold CV, RMSLE)')
print('=' * 60)
results = evaluate_models(X, y)

best_model = min(results, key=lambda k: results[k]['mean'])
print(f'\nベストモデル: {best_model} (RMSLE={results[best_model]["mean"]:.5f})')

## 単体ベストモデルで予測

In [ ]:
def train_and_predict(X, y, X_test, test_ids, model_name='GBR', seed=42):
    """フル学習 → テスト予測 → submission.csv"""
    models = get_models(seed)
    model = models[model_name]
    is_cb = model_name in CATBOOST_MODELS
    pipe = make_pipeline(model, use_catboost=is_cb)

    pipe.fit(X, y)
    preds_log = pipe.predict(X_test)
    preds = np.expm1(preds_log)
    preds = np.maximum(preds, 0)

    submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
    filename = f'submission_{model_name.lower()}.csv'
    submission.to_csv(filename, index=False)
    print(f'\n{filename} を作成しました ({len(submission)}行)')
    print(f'SalePrice: mean={preds.mean():.0f}, median={np.median(preds):.0f}')

    return submission

train_and_predict(X, y, X_test, test_ids, model_name=best_model)

## OOF スタッキング

In [ ]:
def stacking_predict(X, y, X_test, test_ids, seed=42):
    """OOFスタッキング: 複数モデル → Ridgeメタモデル"""
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    base_models = get_models(seed)

    n_train = len(X)
    n_test = len(X_test)
    n_models = len(base_models)

    oof_preds = np.zeros((n_train, n_models))
    test_preds = np.zeros((n_test, n_models))

    for i, (name, model) in enumerate(base_models.items()):
        print(f'  {name}...', end='', flush=True)
        test_preds_fold = np.zeros((n_test, 5))
        is_cb = name in CATBOOST_MODELS

        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            pipe = make_pipeline(model, use_catboost=is_cb)
            pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
            oof_preds[val_idx, i] = pipe.predict(X.iloc[val_idx])
            test_preds_fold[:, fold] = pipe.predict(X_test)

        test_preds[:, i] = test_preds_fold.mean(axis=1)
        rmse = np.sqrt(mean_squared_error(y, oof_preds[:, i]))
        print(f' OOF RMSLE={rmse:.5f}')

    # メタモデル
    meta = Ridge(alpha=1.0)
    meta.fit(oof_preds, y)
    meta_pred = meta.predict(test_preds)

    # OOF CVスコア
    meta_cv = np.sqrt(-cross_val_score(
        Ridge(alpha=1.0), oof_preds, y,
        cv=KFold(5, shuffle=True, random_state=seed),
        scoring='neg_mean_squared_error'))
    print(f'\nStack CV RMSLE = {meta_cv.mean():.5f} ± {meta_cv.std():.5f}')
    print(f'Meta weights: {dict(zip(base_models.keys(), meta.coef_.round(3)))}')

    preds = np.expm1(meta_pred)
    preds = np.maximum(preds, 0)

    submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
    submission.to_csv('submission_stack.csv', index=False)
    print(f'\nsubmission_stack.csv を作成しました ({len(submission)}行)')
    print(f'SalePrice: mean={preds.mean():.0f}, median={np.median(preds):.0f}')

    return submission, meta_cv.mean()

print('=' * 60)
print('スタッキング')
print('=' * 60)
stacking_predict(X, y, X_test, test_ids)

## マルチシード スタッキング (Seed Averaging)

In [ ]:
print('=' * 60)
print('マルチシード スタッキング (seeds: 42, 123, 456)')
print('=' * 60)
multi_preds = []
for seed in [42, 123, 456]:
    print(f'\n--- seed={seed} ---')
    sub, cv = stacking_predict(X, y, X_test, test_ids, seed=seed)
    multi_preds.append(sub['SalePrice'].values)

avg_preds = np.mean(multi_preds, axis=0)
submission_multi = pd.DataFrame({'Id': test_ids, 'SalePrice': avg_preds})
submission_multi.to_csv('submission_multiseed.csv', index=False)
print(f'\nsubmission_multiseed.csv を作成しました')
print(f'SalePrice: mean={avg_preds.mean():.0f}, median={np.median(avg_preds):.0f}')

## 提出ファイルダウンロード (Colab用)

In [ ]:
if IS_COLAB:
    from google.colab import files
    files.download('submission_multiseed.csv')
    files.download('submission_stack.csv')
else:
    print("Local: submission files saved in current directory.")
    !ls -la submission_*.csv